In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import time

from src.data_loader import (load_ratings, train_test_split_by_time,
                              build_interaction_matrix, get_user_positive_items)
from src.metrics import evaluate_rankings
from src.utils import set_seed
from src.models.popularity import PopularityRecommender
from src.models.knn_cf import UserBasedCF, ItemBasedCF
from src.models.mf import SVDRecommender

In [ ]:
set_seed(42)
ratings = load_ratings('../data/raw')
train, test = train_test_split_by_time(ratings, test_ratio=0.2)
print(f'train: {len(train)}, test: {len(test)}')

In [ ]:
K = 10
THRESHOLD = 4

# ground truth
test_gt = {}
for uid, group in test.groupby('user_id'):
    pos = group[group['rating'] >= THRESHOLD]['item_id'].tolist()
    if pos:
        test_gt[uid] = pos

train_user_items = {}
for uid, group in train.groupby('user_id'):
    train_user_items[uid] = set(group['item_id'].values)

print(f'{len(test_gt)} users with positive items in test set')

## Popularity baseline

In [ ]:
pop = PopularityRecommender()
pop.fit(train)
pop_recs = pop.recommend_all(test_gt.keys(), train_user_items, n=K)
pop_metrics = evaluate_rankings(pop_recs, test_gt, K)
print('Popularity:', pop_metrics)

## User-based CF

In [ ]:
mat = build_interaction_matrix(train, 943, 1682)

# 0-indexed exclude sets
train_items_idx = {}
for _, row in train.iterrows():
    uid_idx = int(row['user_id']) - 1
    iid_idx = int(row['item_id']) - 1
    if uid_idx not in train_items_idx:
        train_items_idx[uid_idx] = set()
    train_items_idx[uid_idx].add(iid_idx)

In [ ]:
t0 = time.time()
ubcf = UserBasedCF(k=20)
ubcf.fit(mat)
ubcf_recs = ubcf.recommend_all_users(943, train_items_idx, n=K)
ubcf_recs = {u: ubcf_recs[u] for u in test_gt if u in ubcf_recs}
ubcf_metrics = evaluate_rankings(ubcf_recs, test_gt, K)
print(f'UserCF (k=20): {ubcf_metrics}  [{time.time()-t0:.1f}s]')

## Item-based CF

This is slow since we compute scores per-item for each user.

In [ ]:
# item-based is slow on full dataset, run on subset for demo
# uncomment below to run on all users (takes ~10min)

ibcf = ItemBasedCF(k=20)
ibcf.fit(mat)

# just evaluate on 100 random users for speed
sample_users = list(test_gt.keys())[:100]
ibcf_recs = {}
for uid in sample_users:
    uid_idx = uid - 1
    exclude = train_items_idx.get(uid_idx, set())
    recs = ibcf.recommend(uid_idx, n=K, exclude_items=exclude)
    ibcf_recs[uid] = [r + 1 for r in recs]

ibcf_gt = {u: test_gt[u] for u in sample_users if u in test_gt}
ibcf_metrics = evaluate_rankings(ibcf_recs, ibcf_gt, K)
print(f'ItemCF (k=20, 100 users): {ibcf_metrics}')

## SVD matrix factorization

In [ ]:
t0 = time.time()
svd = SVDRecommender(n_factors=50)
svd.fit(mat)
svd_recs = svd.recommend_all_users(943, train_items_idx, n=K)
svd_recs = {u: svd_recs[u] for u in test_gt if u in svd_recs}
svd_metrics = evaluate_rankings(svd_recs, test_gt, K)
print(f'SVD (k=50): {svd_metrics}  [{time.time()-t0:.1f}s]')

In [ ]:
# try different number of factors
for nf in [10, 30, 50, 100]:
    svd_tmp = SVDRecommender(n_factors=nf)
    svd_tmp.fit(mat)
    recs = svd_tmp.recommend_all_users(943, train_items_idx, n=K)
    recs = {u: recs[u] for u in test_gt if u in recs}
    m = evaluate_rankings(recs, test_gt, K)
    print(f'  n_factors={nf}: HR={m["HR@10"]:.4f}, NDCG={m["NDCG@10"]:.4f}')

## Summary

In [ ]:
all_results = {
    'Popularity': pop_metrics,
    'UserCF': ubcf_metrics,
    'SVD': svd_metrics,
}

for name, m in all_results.items():
    print(f"{name:15s}  HR={m['HR@10']:.4f}  NDCG={m['NDCG@10']:.4f}  "
          f"Prec={m['Precision@10']:.4f}  Recall={m['Recall@10']:.4f}")